In [1]:
from huggingface_hub import snapshot_download
model_path = 'Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B'
# model_path = 'Qwen/Qwen2.5-Math-PRM-7B'
# model_path = 'Qwen/Qwen2.5-7B-Instruct'
snapshot_download(repo_id=model_path, local_dir=f"hf_cache/{'--'.join(model_path.split('/'))}")

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

configuration_qwen2_rm.py:   0%|          | 0.00/6.68k [00:00<?, ?B/s]

.gitattributes:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.33k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

'/home/udbhavbamba/Projects/prm-inference/hf_cache/Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B'

In [ ]:
from openai import OpenAI
from transformers import AutoTokenizer
from model_utils.io_utils import prepare_input, derive_step_rewards_vllm, prepare_batch_input_for_model

In [2]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8081/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
models = client.models.list()
model = models.data[0].id

In [3]:
data_list = [
    {
        "problem": "Cindy's math and science books weigh 2 pounds each. Her French book weighs 4 pounds and her English book weighs 3 pounds. Her history book weighs twice as much as her English book. If Cindy carries all of her books at once, what will be the total weight of the books she is carrying?",
        "steps": [
            "To determine the total weight of all Cindy's books, we need to calculate the weight of each book individually and then sum these weights.", 
            "First, for the math and science books:\n- Each math book weighs 2 pounds.\n- Each science book weighs 2 pounds.\n- Cindy has 2 math books and 2 science books.\n- Total weight of math books: \\(2 \\text{ books} \\times 2 \\text{ pounds/book} = 4 \\text{ pounds}\\).\n- Total weight of science books: \\(2 \\text{ books} \\times 2 \\text{ pounds/book} = 4 \\text{ pounds}\\).\n- Combined weight of math and science books: \\(4 \\text{ pounds} + 4 \\text{ pounds} = 8 \\text{ pounds}\\).", 
            "Second, for the French book:\n- The French book weighs 4 pounds.",
            "Third, for the English book:\n- The English book weighs 3 pounds.", 
            "Fourth, for the history book:\n- The history book weighs twice as much as the English book.\n- Weight of the history book: \\(2 \\times 3 \\text{ pounds} = 6 \\text{ pounds}\\).",
            "Finally, for the total weight:\n- Sum of the weights of all the books: \\[ 8 \\text{ pounds} \\text{ (math and science)} + 4 \\text{ pounds} \\text{ (French)} + 3 \\text{ pounds} \\text{ (English)} + 6 \\text{ pounds} \\text{ (history)} = 21 \\text{ pounds} \\]",
            "Therefore, the total weight of the books Cindy is carrying is \\(\\boxed{21}\\) pounds."
        ]
    },
    {
        "problem": "Sue lives in a fun neighborhood.  One weekend, the neighbors decided to play a prank on Sue.  On Friday morning, the neighbors placed 18 pink plastic flamingos out on Sue's front yard.  On Saturday morning, the neighbors took back one third of the flamingos, painted them white, and put these newly painted white flamingos back out on Sue's front yard.  Then, on Sunday morning, they added another 18 pink plastic flamingos to the collection. At noon on Sunday, how many more pink plastic flamingos were out than white plastic flamingos?",
        "steps": [
            "To find out how many more pink plastic flamingos were out than white plastic flamingos at noon on Sunday, we can break down the problem into steps. First, on Friday, the neighbors start with 18 pink plastic flamingos.",
            "On Saturday, they take back one third of the flamingos. Since there were 18 flamingos, (1/3 \\times 18 = 6) flamingos are taken back. So, they have (18 - 6 = 12) flamingos left in their possession. Then, they paint these 6 flamingos white and put them back out on Sue's front yard. Now, Sue has the original 12 pink flamingos plus the 6 new white ones. Thus, by the end of Saturday, Sue has (12 + 6 = 18) pink flamingos and 6 white flamingos.",
            "On Sunday, the neighbors add another 18 pink plastic flamingos to Sue's front yard. By the end of Sunday morning, Sue has (18 + 18 = 36) pink flamingos and still 6 white flamingos.",
            "To find the difference, subtract the number of white flamingos from the number of pink flamingos: (36 - 6 = 30). Therefore, at noon on Sunday, there were 30 more pink plastic flamingos out than white plastic flamingos. The answer is (\\boxed{30})."
        ]
    }
]

### Skywork/Skywork-o1-Open-PRM-Qwen-2.5-1.5B

```
vllm serve hf_cache/Skywork--Skywork-o1-Open-PRM-Qwen-2.5-1.5B \
    --host 0.0.0.0 \
    --port 8081 \
    --gpu-memory-utilization 0.9 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [4]:
SKYWORK_MODEL_PATH = "hf_cache/Skywork--Skywork-o1-Open-PRM-Qwen-2.5-1.5B"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(SKYWORK_MODEL_PATH)

input_ids_batch = []
token_mask_batch = []
for data in data_list:
    input_ids, token_mask = prepare_input(
                            SKYWORK_MODEL_PATH, 
                            problem=data["problem"], 
                            steps=data["steps"], 
                            tokenizer=tokenizer,
                            convert_to_list=True
    )
    input_ids_batch.append(input_ids)
    token_mask_batch.append(token_mask)

input_ids, token_masks = prepare_batch_input_for_model(input_ids_batch, token_mask_batch, pad_token_id=0)
logits = client.embeddings.create(
    input=input_ids.cpu().tolist(),
    model=model,
)
rewards = derive_step_rewards_vllm(
    SKYWORK_MODEL_PATH, 
    logits, 
    token_masks, 
    tokenizer
)
print(rewards)

[[0.9314625069680097, 0.5107405350179702, 0.6099889154581032, 0.52122746695113, 0.57415601726698, 0.7892336936802291, 0.8872045937171068], [0.5921950051775453, 0.4201236971273697, 0.5498823399224035, 0.7592254042296177]]


### Qwen/Qwen2.5-Math-PRM-7B

```
vllm serve hf_cache/Qwen--Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8081 \
    --gpu-memory-utilization 0.9 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [4]:
QWEN_MODEL_PATH = "hf_cache/Qwen--Qwen2.5-Math-PRM-7B"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_PATH)

input_ids_batch = []
token_mask_batch = []
for data in data_list:
    input_ids, token_mask = prepare_input(
                            QWEN_MODEL_PATH, 
                            problem=data["problem"], 
                            steps=data["steps"], 
                            tokenizer=tokenizer,
                            convert_to_list=True
    )
    input_ids_batch.append(input_ids)
    token_mask_batch.append(token_mask)

input_ids, token_masks = prepare_batch_input_for_model(input_ids_batch, token_mask_batch, pad_token_id=0)
logits = client.embeddings.create(
    input=input_ids.cpu().tolist(),
    model=model,
)
rewards = derive_step_rewards_vllm(
    QWEN_MODEL_PATH, 
    logits, 
    token_masks, 
    tokenizer
)
print(rewards)

[[1.0, 0.0157470703125, 0.99609375, 1.0, 0.99609375, 0.97265625, 1.0], [1.0, 0.15625, 0.9765625, 1.0]]
